<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/Allison/LightGBM_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**FULL DATA SET FILE:** (model_df_clf_v2.feather)


*   Accuracy: 0.645
*   F1 Score: 0.636

Classification Report:
              precision,    recall,  f1-score,   support

        long       0.69      0.71      0.70    239774
      medium       0.52      0.46      0.49    261451
       short       0.70      0.74      0.72    320908


**PREPROCESSING DATA SET FILE:** (sparcs_clean_v2.feather)


*   Accuracy:
*   F1 Score:





In [18]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

drive.mount('/content/drive')
path = "/content/drive/MyDrive/sparcs_clean_v2.feather"
data = pd.read_feather(path)

data.info()

#model_df_clf_v2.feather
#sparcs_encoded.feather

data['length_of_stay'] = pd.to_numeric(data['length_of_stay'], errors='coerce')
data = data.dropna(subset=['length_of_stay'])
data = data[data['zip_code'] != 'OOS']


#Features
categorical_cols = [
    'health_service_area', 'zip_code', 'hospital_county','age_group',
    'gender', 'race', 'ethnicity', 'admission_type', 'apr_mortality_risk',
    'apr_med_surg_desc', 'ccsr_dx_code', 'ccsr_px_code', 'Payment Typology 1' ]

numeric_cols = ["facility_id", "apr_drg_code", "apr_mdc_code", "apr_severity_code", "emergency_dept_indicator"]

feature_cols = categorical_cols + numeric_cols

for col in categorical_cols:
    data[col] = data[col].astype('category')




#Classification Targets
#LOS 3-class (short/medium/long)
data['los_3class'] = pd.qcut(data['length_of_stay'], q=3, labels=['short', 'medium', 'long']).astype('category')

#LOS binary classification (short vs long)
data['los_binary'] = pd.qcut(data['length_of_stay'], q=2,labels=['short', 'long']).astype('category')




#Train & Evaluate Classifiers
targets = {"LOS (3-class)": "los_3class", "LOS (binary)": "los_binary"}

results = []


for label, target_col in targets.items():
    print(f"\nTraining Classifier for: {label}")

    # Convert binary labels to numeric 0/1
    if target_col == "los_binary":
        data[target_col] = data[target_col].map({"short": 0, "long": 1})

    X = data[feature_cols]
    y = data[target_col]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Select correct LightGBM objective
    if target_col == "los_3class":
        objective = "multiclass"
        num_class = 3
    else:
        objective = "binary"
        num_class = 1

    model = lgb.LGBMClassifier(
        objective=objective,
        num_class=num_class,
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Metrics
    if target_col == "los_binary":
        # binary
        f1_macro = f1_score(y_test, y_pred)
    else:
        # multi-class
        f1_macro = f1_score(y_test, y_pred, average="macro")

    acc = accuracy_score(y_test, y_pred)

    print(f"Accuracy: {acc:.3f}")
    print(f"Macro F1: {f1_macro:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
Index: 2597703 entries, 0 to 4322489
Data columns (total 19 columns):
 #   Column                    Dtype 
---  ------                    ----- 
 0   health_service_area       object
 1   hospital_county           object
 2   facility_id               int64 
 3   age_group                 object
 4   zip_code                  object
 5   gender                    object
 6   race                      object
 7   ethnicity                 object
 8   length_of_stay            int64 
 9   admission_type            object
 10  ccsr_dx_code              object
 11  ccsr_px_code              object
 12  apr_drg_code              int64 
 13  apr_mdc_code              int64 
 14  apr_severity_code         int64 
 15  apr_mortality_risk        object
 16  apr_med_surg_desc         object
 17  Payment Typology 1        object
 18

In [19]:
#Saving Model
model.booster_.save_model("/content/drive/MyDrive/lightgbm_classifier_model.txt")